In [2]:
import json
import re
import pandas as pd
from pathlib import Path
import os
from scipy import stats
import numpy as np

In [3]:
USED_QUESTS = [
    "claude-haiku-4-5-automata_v2-delivery_v2/quest1",
    "claude-haiku-4-5-automata_v2-delivery_v2/quest4",
    "claude-haiku-4-5-automata_v2-delivery_v2/quest9",
    "claude-haiku-4-5-automata_v2-delivery_v2/quest11",
    "claude-haiku-4-5-automata_v2-delivery_v2/quest15",
    
    "deepseek-v3.2-automata_v2-delivery_v2/quest3",
    "deepseek-v3.2-automata_v2-delivery_v2/quest6",
    "deepseek-v3.2-automata_v2-delivery_v2/quest10",
    "deepseek-v3.2-automata_v2-delivery_v2/quest13",
    "deepseek-v3.2-automata_v2-delivery_v2/quest14",
    
    "gemini-3-flash-preview-automata_v2-delivery_v2/quest1",
    "gemini-3-flash-preview-automata_v2-delivery_v2/quest4",
    "gemini-3-flash-preview-automata_v2-delivery_v2/quest5",
    "gemini-3-flash-preview-automata_v2-delivery_v2/quest12",
    "gemini-3-flash-preview-automata_v2-delivery_v2/quest15",
    
    "gpt-5.4-automata_v2-delivery_v2/quest1",
    "gpt-5.4-automata_v2-delivery_v2/quest7",
    "gpt-5.4-automata_v2-delivery_v2/quest10",
    "gpt-5.4-automata_v2-delivery_v2/quest14",
    "gpt-5.4-automata_v2-delivery_v2/quest15",
]

In [4]:
def quest_json_to_row(filepath: str) -> dict:
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    row = {
        "quest": data["quest"],
        "estimator": data["estimator"].split("/")[-1],
    }

    for metric_key, metric_value in data.get("metrics", {}).items():
        match = re.search(r"\(([^)]+)\)", metric_key)
        column_name = match.group(1) if match else metric_key
        row[column_name] = metric_value.get("score")

    return row

def get_estimation_json_paths(root_dir: str = "./results/estimation") -> list[str]:
    return [str(path) for path in Path(root_dir).rglob("*.json") if path.is_file()]

def estimation_jsons_to_rows(filepaths: list[str]) -> list[dict]:
    return [quest_json_to_row(filepath) for filepath in filepaths]

In [5]:
def corr_spearman(x, y):
    res = stats.spearmanr(x, y, nan_policy="omit", alternative="two-sided")
    return res.statistic, res.pvalue

def corr_kendall(x, y):
    res = stats.kendalltau(x, y, nan_policy="omit", alternative="two-sided", method="auto")
    return res.statistic, res.pvalue

In [6]:
def perm_test_spearman(x, y, n_resamples=50_000, permutation_type="pairings"):
    x = np.asarray(x)
    y = np.asarray(y)

    def stat(a, b, axis=-1):
        return stats.spearmanr(a, b, nan_policy="omit").statistic

    res = stats.permutation_test(
        data=(x, y),
        statistic=stat,
        permutation_type=permutation_type,
        alternative="two-sided",
        n_resamples=n_resamples,
        vectorized=False,
    )
    return res.statistic, res.pvalue


def perm_test_kendall(x, y, n_resamples=50_000, permutation_type="pairings"):
    x = np.asarray(x)
    y = np.asarray(y)

    def stat(a, b, axis=-1):
        return stats.kendalltau(a, b, nan_policy="omit", method="auto").statistic

    res = stats.permutation_test(
        data=(x, y),
        statistic=stat,
        permutation_type=permutation_type,
        alternative="two-sided",
        n_resamples=n_resamples,
        vectorized=False,
    )
    return res.statistic, res.pvalue

In [7]:
def generate_corr_dict(stat, p_value):
    return {
        "stat": f"{stat:.6f}",
        "p_value": f"{p_value:.6f}"
    }


def generate_corr(x, y, stat_func, perm_stat_func):
    stat, p_value = stat_func(x, y)
    perm_stat, perm_p_value = perm_stat_func(x, y)
    return {
        "asymptotic": generate_corr_dict(stat, p_value),
        "permutation_test": generate_corr_dict(perm_stat, perm_p_value)
    }


def generate_all_corrs(x, y):
    return {
        "spearmanr": generate_corr(x, y, corr_spearman, perm_test_spearman),
        "kendall": generate_corr(x, y, corr_kendall, perm_test_kendall)
    }


def generate_per_criterion_corrs(criteria_cols, h_quests, human_estimation, quests, llm_estimation_df):
    results = []

    for criterion in criteria_cols:
        print(criterion)
        x = []
        x_keys = []

        for q in h_quests:
            data = human_estimation[human_estimation['quest'] == q][criterion].median()
            x.append(data)
            x_keys.append((criterion, q))
        
        y = []
        y_keys = []

        for q in quests:
            data = llm_estimation_df[llm_estimation_df['quest'] == q].iloc[0][criterion]
            y.append(data)
            y_keys.append((criterion, q))
        
        results.append({
            "criterion": criterion,
            "spearmanr": generate_corr(x, y, corr_spearman, perm_test_spearman),
            "kendall": generate_corr(x, y, corr_kendall, perm_test_kendall)
        })
    
    return results


def generate_corr_json(dialog_criterion, aggregated_dialog_medians, criterion):
    return {
        "dialog_criterion": dialog_criterion,
        "aggregated_dialog_medians": aggregated_dialog_medians,
        "criterion": criterion
    }

In [8]:
filepaths = get_estimation_json_paths("../results/estimation")
filepaths = list(filter(lambda p: p.endswith('_1.json') and "estimated_by_openai\\gpt-5.4-mini" in p, filepaths))

In [9]:
filepaths

['..\\results\\estimation\\gpt-5.4-automata_v2-delivery_v2\\estimated_by_openai\\gpt-5.4-mini\\gpt-5.4-automata_v2-delivery_v2\\quest10_1.json',
 '..\\results\\estimation\\gpt-5.4-automata_v2-delivery_v2\\estimated_by_openai\\gpt-5.4-mini\\gpt-5.4-automata_v2-delivery_v2\\quest14_1.json',
 '..\\results\\estimation\\gpt-5.4-automata_v2-delivery_v2\\estimated_by_openai\\gpt-5.4-mini\\gpt-5.4-automata_v2-delivery_v2\\quest15_1.json',
 '..\\results\\estimation\\gpt-5.4-automata_v2-delivery_v2\\estimated_by_openai\\gpt-5.4-mini\\gpt-5.4-automata_v2-delivery_v2\\quest1_1.json',
 '..\\results\\estimation\\gpt-5.4-automata_v2-delivery_v2\\estimated_by_openai\\gpt-5.4-mini\\gpt-5.4-automata_v2-delivery_v2\\quest7_1.json',
 '..\\results\\estimation\\gemini-3-flash-preview-automata_v2-delivery_v2\\estimated_by_openai\\gpt-5.4-mini\\gemini-3-flash-preview-automata_v2-delivery_v2\\quest12_1.json',
 '..\\results\\estimation\\gemini-3-flash-preview-automata_v2-delivery_v2\\estimated_by_openai\\gpt-5.

In [10]:
rows = estimation_jsons_to_rows(filepaths)
llm_estimation_df = pd.DataFrame(rows)

In [11]:
llm_estimation_df.head()

,quest,estimator,relevancy,coherence,naturalness,faithfulness,safety,unbiasedness,non-toxicity
0,gpt-5.4-automata_v2-delivery_v2/quest10_1,gpt-5.4-mini,0.3,0.4,0.3,0.4,1.0,1.0,1.0
1,gpt-5.4-automata_v2-delivery_v2/quest14_1,gpt-5.4-mini,0.5,0.8,0.2,0.2,1.0,1.0,1.0
2,gpt-5.4-automata_v2-delivery_v2/quest15_1,gpt-5.4-mini,0.4,0.4,0.3,0.4,0.9,1.0,0.8
3,gpt-5.4-automata_v2-delivery_v2/quest1_1,gpt-5.4-mini,0.9,0.8,0.8,0.9,1.0,0.9,0.7
4,gpt-5.4-automata_v2-delivery_v2/quest7_1,gpt-5.4-mini,0.9,0.8,0.5,0.9,1.0,1.0,0.4


In [12]:
llm_estimation_df[llm_estimation_df.select_dtypes(include="number").columns] *= 10

In [13]:
human_estimation = pd.read_csv('../notebooks/results_24-04-2026_2_dataframe.csv')

In [14]:
human_estimation = human_estimation.drop(columns=["file_name", "file_path", "comment"], errors="ignore")
human_estimation.columns = [
    re.search(r"\(([^()]+)\)", col).group(1) if isinstance(col, str) and re.search(r"\(([^()]+)\)", col) else col
    for col in human_estimation.columns
]

human_estimation = human_estimation[human_estimation["quest"].isin(USED_QUESTS)]


In [15]:
len(human_estimation)

169

In [16]:
human_estimation.head()

,quest,relevancy,coherence,naturalness,faithfulness,safety,unbiasedness,non-toxicity,session_id
0,deepseek-v3.2-automata_v2-delivery_v2/quest6,10,9,9,7,10,10,10,NaN
1,gpt-5.4-automata_v2-delivery_v2/quest15,9,5,9,9,10,10,10,NaN
2,claude-haiku-4-5-automata_v2-delivery_v2/quest4,6,8,9,7,10,10,10,NaN
3,gpt-5.4-automata_v2-delivery_v2/quest15,10,10,9,10,10,9,10,NaN
4,gemini-3-flash-preview-automata_v2-delivery_v2...,8,10,9,10,10,10,8,NaN


## Диалог-критерий

x[i] = медиана по людям для 1 диалога и 1 критерия

y[i] = оценка LLM  для этого же диалога и этого же критерия

In [84]:
criteria_cols = ['relevancy', 'coherence', 'naturalness', 'faithfulness', 'safety', 'unbiasedness', 'non-toxicity']


In [85]:
h_quests = sorted(set(human_estimation['quest']))

In [86]:
h_quests

['claude-haiku-4-5-automata_v2-delivery_v2/quest1',
 'claude-haiku-4-5-automata_v2-delivery_v2/quest11',
 'claude-haiku-4-5-automata_v2-delivery_v2/quest15',
 'claude-haiku-4-5-automata_v2-delivery_v2/quest4',
 'claude-haiku-4-5-automata_v2-delivery_v2/quest9',
 'deepseek-v3.2-automata_v2-delivery_v2/quest10',
 'deepseek-v3.2-automata_v2-delivery_v2/quest13',
 'deepseek-v3.2-automata_v2-delivery_v2/quest14',
 'deepseek-v3.2-automata_v2-delivery_v2/quest3',
 'deepseek-v3.2-automata_v2-delivery_v2/quest6',
 'gemini-3-flash-preview-automata_v2-delivery_v2/quest1',
 'gemini-3-flash-preview-automata_v2-delivery_v2/quest12',
 'gemini-3-flash-preview-automata_v2-delivery_v2/quest15',
 'gemini-3-flash-preview-automata_v2-delivery_v2/quest4',
 'gemini-3-flash-preview-automata_v2-delivery_v2/quest5',
 'gpt-5.4-automata_v2-delivery_v2/quest1',
 'gpt-5.4-automata_v2-delivery_v2/quest10',
 'gpt-5.4-automata_v2-delivery_v2/quest14',
 'gpt-5.4-automata_v2-delivery_v2/quest15',
 'gpt-5.4-automata_v2-d

In [87]:
quests = [s + "_1" for s in h_quests]
quests = ["deepseek/" + s if s.startswith("deepseek") else s for s in quests]

In [88]:
quests

['claude-haiku-4-5-automata_v2-delivery_v2/quest1_1',
 'claude-haiku-4-5-automata_v2-delivery_v2/quest11_1',
 'claude-haiku-4-5-automata_v2-delivery_v2/quest15_1',
 'claude-haiku-4-5-automata_v2-delivery_v2/quest4_1',
 'claude-haiku-4-5-automata_v2-delivery_v2/quest9_1',
 'deepseek/deepseek-v3.2-automata_v2-delivery_v2/quest10_1',
 'deepseek/deepseek-v3.2-automata_v2-delivery_v2/quest13_1',
 'deepseek/deepseek-v3.2-automata_v2-delivery_v2/quest14_1',
 'deepseek/deepseek-v3.2-automata_v2-delivery_v2/quest3_1',
 'deepseek/deepseek-v3.2-automata_v2-delivery_v2/quest6_1',
 'gemini-3-flash-preview-automata_v2-delivery_v2/quest1_1',
 'gemini-3-flash-preview-automata_v2-delivery_v2/quest12_1',
 'gemini-3-flash-preview-automata_v2-delivery_v2/quest15_1',
 'gemini-3-flash-preview-automata_v2-delivery_v2/quest4_1',
 'gemini-3-flash-preview-automata_v2-delivery_v2/quest5_1',
 'gpt-5.4-automata_v2-delivery_v2/quest1_1',
 'gpt-5.4-automata_v2-delivery_v2/quest10_1',
 'gpt-5.4-automata_v2-delivery_v

In [89]:
agg_medians = (
    human_estimation
    .groupby("quest", sort=False)[criteria_cols]
    .median(numeric_only=True)
)

x = []
x_keys = []

for q in h_quests:
    if q not in agg_medians.index:
        continue
    for criterion in criteria_cols:
        x.append(agg_medians.loc[q, criterion])
        x_keys.append((q, criterion))

In [90]:
y = []
y_keys = []

for q in quests:
    row = llm_estimation_df.loc[llm_estimation_df['quest'] == q]
    if row.empty:
        continue

    for criterion in criteria_cols:
        y.append(row.iloc[0][criterion])
        y_keys.append((q, criterion))


In [91]:
dialogs_criterion = generate_all_corrs(x ,y)
dialogs_criterion

{'spearmanr': {'asymptotic': {'stat': '0.644556', 'p_value': '0.000000'},
  'permutation_test': {'stat': '0.644556', 'p_value': '0.000040'}},
 'kendall': {'asymptotic': {'stat': '0.536095', 'p_value': '0.000000'},
  'permutation_test': {'stat': '0.536095', 'p_value': '0.000040'}}}

## Агрегация по диалогам (медиана)

In [92]:
x = []
x_keys = []

for q in h_quests:
    x.append(human_estimation[human_estimation['quest'] == q][criteria_cols].median(axis=None))
    x_keys.append(q)

In [93]:
y = []
y_keys = []

for q in quests:
    y.append(llm_estimation_df[llm_estimation_df['quest'] == q][criteria_cols].median(axis=None))
    y_keys.append(q)

In [94]:
aggregated_dialog_medians = generate_all_corrs(x ,y)
aggregated_dialog_medians

{'spearmanr': {'asymptotic': {'stat': '0.199795', 'p_value': '0.398366'},
  'permutation_test': {'stat': '0.199795', 'p_value': '0.376632'}},
 'kendall': {'asymptotic': {'stat': '0.179200', 'p_value': '0.383815'},
  'permutation_test': {'stat': '0.179200', 'p_value': '0.373513'}}}

## То же, но сумма

In [115]:
x = []
x_keys = []

for q in h_quests:
    x.append(human_estimation[human_estimation['quest'] == q][criteria_cols].sum(axis=1).median())
    x_keys.append(q)

In [ ]:
y = []
y_keys = []

for q in quests:
    y.append(llm_estimation_df[llm_estimation_df['quest'] == q][criteria_cols].sum(axis=1).median())
    y_keys.append(q)

In [ ]:
out = {}
out["spearman"] = corr_spearman(x, y)
out["kendall"]  = corr_kendall(x, y)
out["perm_spearman"] = perm_test_spearman(x, y, n_resamples=50_000)
out["perm_kendall"]  = perm_test_kendall(x, y, n_resamples=50_000)

out

{'spearman': (np.float64(0.5168242335343465),
  np.float64(0.019630532293384087)),
 'kendall': (np.float64(0.39782164030610784),
  np.float64(0.017022470639666647)),
 'perm_spearman': (np.float64(0.5168242335343465),
  np.float64(0.020799584008319834)),
 'perm_kendall': (np.float64(0.39782164030610784),
  np.float64(0.01723965520689586))}

## По критериям

In [95]:
criterion_summary = generate_per_criterion_corrs(criteria_cols, h_quests, human_estimation, quests, llm_estimation_df)
criterion_summary

relevancy
coherence
naturalness
faithfulness
safety
unbiasedness


C:\Users\dlyko\AppData\Local\Temp\ipykernel_17652\1381373193.py:2: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  res = stats.spearmanr(x, y, nan_policy="omit", alternative="two-sided")
C:\Users\dlyko\AppData\Local\Temp\ipykernel_17652\3640273269.py:6: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return stats.spearmanr(a, b, nan_policy="omit").statistic


non-toxicity


[{'criterion': 'relevancy',
  'spearmanr': {'asymptotic': {'stat': '0.249015', 'p_value': '0.289730'},
   'permutation_test': {'stat': '0.249015', 'p_value': '0.286554'}},
  'kendall': {'asymptotic': {'stat': '0.205560', 'p_value': '0.271207'},
   'permutation_test': {'stat': '0.205560', 'p_value': '0.285994'}}},
 {'criterion': 'coherence',
  'spearmanr': {'asymptotic': {'stat': '0.470772', 'p_value': '0.036172'},
   'permutation_test': {'stat': '0.470772', 'p_value': '0.036439'}},
  'kendall': {'asymptotic': {'stat': '0.385392', 'p_value': '0.042467'},
   'permutation_test': {'stat': '0.385392', 'p_value': '0.042559'}}},
 {'criterion': 'naturalness',
  'spearmanr': {'asymptotic': {'stat': '0.433240', 'p_value': '0.056366'},
   'permutation_test': {'stat': '0.433240', 'p_value': '0.059879'}},
  'kendall': {'asymptotic': {'stat': '0.350177', 'p_value': '0.059741'},
   'permutation_test': {'stat': '0.350177', 'p_value': '0.058879'}}},
 {'criterion': 'faithfulness',
  'spearmanr': {'asymp

In [96]:
import json
with open('./results/corr_24-04-2026_2.json', 'w', encoding='utf-8') as f:
    data = generate_corr_json(dialogs_criterion, aggregated_dialog_medians, criterion_summary)
    json.dump(data, f, ensure_ascii=False, indent=4)

# Результаты

## Диалог-критерий

103 отфильтрованных файла в архиве

```
{'spearman': (np.float64(0.577272359380198),
  np.float64(4.135681052143201e-14)),
 'kendall': (np.float64(0.46159711167940476),
  np.float64(4.489082017939514e-12)),
 'perm_spearman': (np.float64(0.577272359380198),
  np.float64(9.99950002499875e-05)),
 'perm_kendall': (np.float64(0.46159711167940476),
  np.float64(9.99950002499875e-05))}
```

125 отфильтрованных файла в архиве

```
{'spearman': (np.float64(0.6060298656619416),
  np.float64(2.1256691356790205e-15)),
 'kendall': (np.float64(0.5004831390710324),
  np.float64(1.9030024084046886e-13)),
 'perm_spearman': (np.float64(0.6060298656619416),
  np.float64(3.999920001599968e-05)),
 'perm_kendall': (np.float64(0.5004831390710324),
  np.float64(3.999920001599968e-05))}
```

149 отфильтрованных файла в архиве 

```
{'spearman': (np.float64(0.6297891998672062),
  np.float64(7.717477532196398e-17)),
 'kendall': (np.float64(0.5194948677200887),
  np.float64(1.5030820008032662e-14)),
 'perm_spearman': (np.float64(0.6297891998672062),
  np.float64(3.999920001599968e-05)),
 'perm_kendall': (np.float64(0.5194948677200887),
  np.float64(3.999920001599968e-05))}
```

149 отфильтрованных файла в архиве, но только критерии: 

```
sss
```

## Агрегация по диалогам (медиана)

```
{'spearman': (np.float64(0.6000490782018753),
  np.float64(0.005158315512439423)),
 'kendall': (np.float64(0.5371657561995715), np.float64(0.007995181325291907)),
 'perm_spearman': (np.float64(0.6000490782018753),
  np.float64(0.004519909601807964)),
 'perm_kendall': (np.float64(0.5371657561995715),
  np.float64(0.004519909601807964))}
```

149 отфильтрованных файла в архиве 

```
{'spearman': (np.float64(0.45672153561515116),
  np.float64(0.042934537877553164)),
 'kendall': (np.float64(0.39392131873511355), np.float64(0.04857874974049569)),
 'perm_spearman': (np.float64(0.45672153561515116),
  np.float64(0.04495910081798364)),
 'perm_kendall': (np.float64(0.39392131873511355),
  np.float64(0.04531909361812764))}
```

## По критериям

| Метрика | Spearman | Kendall | H0 ? |
|---|---|---|---|
| relevancy | 0.30 (0.19) | 0.23 (0.20) | yes |
| coherence | 0.58 (0.006) | 0.48 (0.01) | no |
| naturalness | 0.52 (0.01) | 0.41 (0.02) | no |
| faithfulness | 0.57 (0.01) | 0.44 (0.01) | no |
| safety | 0.25 (0.26) | 0.25 (0.25) | yes |
| unbiasedness | | | |
| non-toxicity | 0.32 (0.15) | 0.29 (0.12) | yes |

In [ ]:
{'relevancy': {'spearman': (np.float64(0.3044299025546637),
   np.float64(0.19188420126224356)),
  'kendall': (np.float64(0.23946971690731614),
   np.float64(0.20587169465166755)),
  'perm_spearman': (np.float64(0.3044299025546637),
   np.float64(0.19007619847603047)),
  'perm_kendall': (np.float64(0.23946971690731614),
   np.float64(0.2254354912901742))},
 'coherence': {'spearman': (np.float64(0.5862971861414691),
   np.float64(0.00659000225036532)),
  'kendall': (np.float64(0.48187180424864823),
   np.float64(0.01049193768184034)),
  'perm_spearman': (np.float64(0.5862971861414691),
   np.float64(0.007759844803103938)),
  'perm_kendall': (np.float64(0.48187180424864823),
   np.float64(0.007239855202895942))},
 'naturalness': {'spearman': (np.float64(0.5272373540856031),
   np.float64(0.016902192806039634)),
  'kendall': (np.float64(0.4121287811306827), np.float64(0.02165347282199999)),
  'perm_spearman': (np.float64(0.5272373540856031),
   np.float64(0.019439611207775844)),
  'perm_kendall': (np.float64(0.4121287811306827),
   np.float64(0.020519589608207836))},
 'faithfulness': {'spearman': (np.float64(0.5757028369831232),
   np.float64(0.007902473193956235)),
  'kendall': (np.float64(0.44456031421657366),
   np.float64(0.015589818717097616)),
  'perm_spearman': (np.float64(0.5757028369831232),
   np.float64(0.009159816803663927)),
  'perm_kendall': (np.float64(0.44456031421657366),
   np.float64(0.014799704005919881))},
 'safety': {'spearman': (np.float64(0.2595971859817558),
   np.float64(0.26903589742018896)),
  'kendall': (np.float64(0.25110490594257906),
   np.float64(0.25782034830084877)),
  'perm_spearman': (np.float64(0.2595971859817558),
   np.float64(0.7103457930841384)),
  'perm_kendall': (np.float64(0.25110490594257906),
   np.float64(0.6947061058778824))},
 'unbiasedness': {'spearman': (nan, nan),
  'kendall': (np.float64(nan), np.float64(nan)),
  'perm_spearman': (np.float64(nan), np.float64(3.999920001599968e-05)),
  'perm_kendall': (np.float64(nan), np.float64(3.999920001599968e-05))},
 'non-toxicity': {'spearman': (np.float64(0.32755170010138107),
   np.float64(0.15860426298007707)),
  'kendall': (np.float64(0.2948965571464163), np.float64(0.12567941176317365)),
  'perm_spearman': (np.float64(0.32755170010138107),
   np.float64(0.15443691126177475)),
  'perm_kendall': (np.float64(0.2948965571464163),
   np.float64(0.13519729605407893))}}

149 отфильтрованных файла в архиве 

```
{'relevancy': {'spearman': (np.float64(0.26983413165409464),
   np.float64(0.24992392843560687)),
  'kendall': (np.float64(0.2148208058877394), np.float64(0.24184361144232125)),
  'perm_spearman': (np.float64(0.26983413165409464),
   np.float64(0.248315033699326)),
  'perm_kendall': (np.float64(0.2148208058877394),
   np.float64(0.250754984900302))},
 'coherence': {'spearman': (np.float64(0.6090215526415651),
   np.float64(0.004370589756516937)),
  'kendall': (np.float64(0.49681555348464634), np.float64(0.0076460140186786)),
  'perm_spearman': (np.float64(0.6090215526415651),
   np.float64(0.005279894402111958)),
  'perm_kendall': (np.float64(0.49681555348464634),
   np.float64(0.006719865602687947))},
 'naturalness': {'spearman': (np.float64(0.43709419246830333),
   np.float64(0.05396676793839513)),
  'kendall': (np.float64(0.32436089007633573),
   np.float64(0.06965160796595962)),
  'perm_spearman': (np.float64(0.43709419246830333),
   np.float64(0.05471890562188756)),
  'perm_kendall': (np.float64(0.32436089007633573),
   np.float64(0.07255854882902342))},
 'faithfulness': {'spearman': (np.float64(0.6273161281336758),
   np.float64(0.0030697283824871005)),
  'kendall': (np.float64(0.5111043545379566),
   np.float64(0.004705822526948174)),
  'perm_spearman': (np.float64(0.6273161281336758),
   np.float64(0.0037199256014879703)),
  'perm_kendall': (np.float64(0.5111043545379566),
   np.float64(0.0031599368012639748))},
 'safety': {'spearman': (nan, nan),
  'kendall': (np.float64(nan), np.float64(nan)),
  'perm_spearman': (np.float64(nan), np.float64(3.999920001599968e-05)),
  'perm_kendall': (np.float64(nan), np.float64(3.999920001599968e-05))},
 'unbiasedness': {'spearman': (nan, nan),
  'kendall': (np.float64(nan), np.float64(nan)),
  'perm_spearman': (np.float64(nan), np.float64(3.999920001599968e-05)),
  'perm_kendall': (np.float64(nan), np.float64(3.999920001599968e-05))},
 'non-toxicity': {'spearman': (np.float64(0.25541556335644766),
   np.float64(0.27709925168167404)),
  'kendall': (np.float64(0.2397155066535972), np.float64(0.20977105715809463)),
  'perm_spearman': (np.float64(0.25541556335644766),
   np.float64(0.2753144937101258)),
  'perm_kendall': (np.float64(0.2397155066535972),
   np.float64(0.2241955160896782))}}
```